needs to be edited for the sfc 5 group


In [15]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from glob import glob
import matplotlib as mpl

mpl.rcParams['font.family']      = 'Arial'
mpl.rcParams['font.weight']      = 'bold'
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['font.size']        = 25

# =============================================================================
# PATHS
# =============================================================================
MASTER_SHEET_PATH = "housing_food_condition.xlsx"
AUGMENTED_CSV_FOLDER = "all_augmented_csv/results_with_vertices"
OUTPUT_PLOTS_FOLDER = "polar_plots_publication"
os.makedirs(OUTPUT_PLOTS_FOLDER, exist_ok=True)

GROUP_COLS = ['Treatment', 'Housing', 'Food']

# =============================================================================
# DATA LOADING HELPERS
# =============================================================================
def load_master_sheet(master_path):
    master = pd.read_excel(master_path, sheet_name=0)
    master = master.dropna(subset=['Video'])
    keep_cols = ['Video'] + [c for c in GROUP_COLS if c in master.columns]
    master = master[keep_cols].copy()
    master.rename(columns={'Video': 'video'}, inplace=True)
    return master

def get_base_name_from_file(csv_path):
    base = os.path.basename(csv_path)
    if base.startswith("aug_"):
        base = base[4:]
    if base.endswith(".csv"):
        base = base[:-4]
    return base

def extract_video_name_from_filename(basename, master_videos):
    matches = [mv for mv in master_videos if basename.startswith(mv)]
    return max(matches, key=len) if matches else None

# =============================================================================
# COLLECT DATA
# =============================================================================
master = load_master_sheet(MASTER_SHEET_PATH)
master_videos = set(master['video'])

all_csv = glob(os.path.join(AUGMENTED_CSV_FOLDER, '*.csv'))
valid_csv = []
for f in all_csv:
    try:
        df_sample = pd.read_csv(f, nrows=5)
        if 'signed_angle_to_SS' in df_sample.columns:
            valid_csv.append(f)
    except Exception:
        continue

polar_data = []
for csv_path in valid_csv:
    basename = get_base_name_from_file(csv_path)
    video_name = extract_video_name_from_filename(basename, master_videos)
    if video_name is None:
        match = re.search(r'(MVI_\d+_roi\d+)', basename)
        video_name = match.group(1) if match else None
        if not video_name:
            continue

    if video_name in master_videos:
        row = master[master['video'] == video_name].iloc[0]
        group_vals = {gc: row[gc] if pd.notna(row[gc]) else 'unknown' for gc in GROUP_COLS}
    else:
        group_vals = {gc: 'unknown' for gc in GROUP_COLS}

    df = pd.read_csv(csv_path)
    if 'signed_angle_to_SS' not in df.columns:
        continue

    for stim, col in [('SS', 'signed_angle_to_SS'), ('FS', 'signed_angle_to_FS')]:
        angles = df[col].dropna().values
        for ang in angles:
            polar_data.append({
                'video': video_name,
                'stimulus': stim,
                'signed_angle': ang,
                **group_vals
            })

if not polar_data:
    raise RuntimeError("No valid angle data collected.")

polar_df = pd.DataFrame(polar_data)
treatment_df = polar_df[polar_df['Treatment'] != 'unknown'].copy()

# =============================================================================
# PUBLICATION POLAR PLOTS (2x2 GRID ONLY)
# =============================================================================
def plot_treatment_polar_publication(df, output_folder):
    """Create publication-ready polar plots in a 2x2 grid for 4 treatment groups."""
    groups = sorted(df['Treatment'].unique())
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 13), subplot_kw={'projection': 'polar'})
    axes = axes.flatten()
    
    colors = {'SS': '#CC79A7', 'FS': '#009E73'}
    n_bins = 360
    bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    theta = np.linspace(-np.pi, np.pi, n_bins)
    bin_width = 2 * np.pi / n_bins
    
    # Calculate global radial upper limit
    all_counts = []
    for g in groups:
        for stim in ['SS', 'FS']:
            vals = df[(df['Treatment'] == g) & (df['stimulus'] == stim)]['signed_angle'].values
            if len(vals) > 0:
                counts, _ = np.histogram(np.radians(vals), bins=bins)
                all_counts.append(counts.max())
    
    max_count = max(all_counts) if all_counts else 1
    y_limit = np.ceil(max_count / 100) * 100
    if y_limit == 0:
        y_limit = 100
    tick_radii = [v for v in np.arange(0, y_limit + 100, 100) if v > 0]
    
    for idx, g in enumerate(groups[:4]):
        ax = axes[idx]
        for stim, color in colors.items():
            vals = df[(df['Treatment'] == g) & (df['stimulus'] == stim)]['signed_angle'].values
            if len(vals) > 0:
                counts, _ = np.histogram(np.radians(vals), bins=bins)
                ax.bar(theta, counts, width=bin_width, alpha=0.6, 
                       color=color, label=stim, edgecolor='none', linewidth=0)
        
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)
        ax.set_ylim(0, y_limit)
        ax.set_rgrids(radii=tick_radii, labels=[''] * len(tick_radii), angle=0)
        ax.set_yticklabels([])
        
        # Reduced Treatment title font size (16pt)
        ax.set_title(g, fontsize=22, fontweight='bold', pad=18)
        
        # Large angular labels (Theta axis - 16pt)
        ax.tick_params(axis='x', labelsize=22, pad=10)
        ax.grid(True, alpha=0.4, linestyle=':', linewidth=1)
        ax.yaxis.set_major_locator(mticker.FixedLocator(tick_radii))
    
    # Remove unused subplots if fewer than 4 groups exist
    for idx in range(len(groups), 4):
        fig.delaxes(axes[idx])
        
    # Legend placed clear of plot markings in the bottom-right corner
    handles = [plt.Rectangle((0, 0), 1, 1, color=colors['SS'], alpha=0.6),
               plt.Rectangle((0, 0), 1, 1, color=colors['FS'], alpha=0.6)]
    fig.legend(handles, ['SS', 'FS'], loc='lower right', bbox_to_anchor=(0.98, 0.02),
               fontsize=22, frameon=True, edgecolor='black', framealpha=0.9, 
               title="Stimulus", title_fontsize=22)
    
    # Subplot margins adjusted so bottom-right theta ticks do not intersect legend
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.10, right=0.88, top=0.92, hspace=0.30, wspace=0.25)
    
    out_path = os.path.join(output_folder, "treatment_polar_plot_publication_v1.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(out_path.replace('.png', '.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Polar 2x2 plot saved to: {out_path}")

# Run generation
plot_treatment_polar_publication(treatment_df, OUTPUT_PLOTS_FOLDER)

Polar 2x2 plot saved to: polar_plots_publication/treatment_polar_plot_publication_v1.png


In [ ]:
import pandas as pd
import numpy as np

def calculate_orientation_index(df):
    """
    Ensure orientation_index column exists.
    Computes cos(angle) in radians from abs_angle or signed_angle if missing.
    """
    df = df.copy()
    if 'orientation_index' not in df.columns:
        # Cosine of angle in degrees (0 deg -> OI = 1.0; 90 deg -> OI = 0.0; 180 deg -> OI = -1.0)
        angle_col = 'abs_angle' if 'abs_angle' in df.columns else 'signed_angle'
        df['orientation_index'] = np.cos(np.radians(df[angle_col]))
    return df


def build_video_summary(df, group_cols=('Treatment', 'Housing', 'Food')):
    """
    Collapse angle-level data to one row per video x stimulus (long format).

    Returns a DataFrame with angle and orientation index aggregations.
    """
    df = calculate_orientation_index(df)
    group_cols = [c for c in group_cols if c in df.columns]

    summary = (
        df.groupby(['video', 'stimulus'] + group_cols)
        .agg(
            n_angles=('abs_angle', 'count'),
            mean_abs_angle=('abs_angle', 'mean'),
            median_abs_angle=('abs_angle', 'median'),
            std_abs_angle=('abs_angle', 'std'),
            mean_orientation_index=('orientation_index', 'mean'),
            median_orientation_index=('orientation_index', 'median')
        )
        .reset_index()
    )

    return summary


def build_video_summary_wide(df, group_cols=('Treatment', 'Housing', 'Food')):
    """
    Collapse angle-level data to ONE ROW PER VIDEO, with separate
    columns for SS and FS stimuli, including orientation index statistics.
    """
    long_summary = build_video_summary(df, group_cols)
    group_cols = [c for c in group_cols if c in df.columns]

    metrics = [
        'n_angles', 
        'mean_abs_angle', 
        'median_abs_angle', 
        'std_abs_angle',
        'mean_orientation_index', 
        'median_orientation_index'
    ]

    # Pivot stimulus (SS/FS) out into separate columns per metric
    wide = long_summary.pivot(
        index=['video'] + group_cols, 
        columns='stimulus',
        values=metrics
    )

    # Flatten MultiIndex columns: ('mean_orientation_index', 'SS') -> 'mean_orientation_index_SS'
    wide.columns = [f"{metric}_{stim}" for metric, stim in wide.columns]
    wide = wide.reset_index()

    # Reorder columns: id cols first, then SS metrics, then FS metrics
    id_cols = ['video'] + group_cols
    ss_cols = [c for c in wide.columns if c.endswith('_SS')]
    fs_cols = [c for c in wide.columns if c.endswith('_FS')]
    wide = wide[id_cols + ss_cols + fs_cols]

    return wide


if __name__ == "__main__":
    # Example usage inside pipeline:
    video_summary_wide_df = build_video_summary_wide(treatment_df)

    out_path = "video_summary_abs_angle_wide.csv"
    video_summary_wide_df.to_csv(out_path, index=False)
    print(f"Saved video-level summary ({len(video_summary_wide_df)} rows) to: {out_path}")
    print(video_summary_wide_df.head())

In [16]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from glob import glob
import matplotlib as mpl

mpl.rcParams['font.family']      = 'Arial'
mpl.rcParams['font.weight']      = 'bold'
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['font.size']        = 25

# =============================================================================
# PATHS
# =============================================================================
MASTER_SHEET_PATH = "housing_food_condition.xlsx"
AUGMENTED_CSV_FOLDER = "all_augmented_csv/results_with_vertices"
OUTPUT_PLOTS_FOLDER = "polar_plots_publication"
os.makedirs(OUTPUT_PLOTS_FOLDER, exist_ok=True)

GROUP_COLS = ['Treatment', 'Housing', 'Food']

# =============================================================================
# PARAMETERS
# =============================================================================
FRAME_RATE = 25.0        # frames per second

# =============================================================================
# DATA LOADING HELPERS
# =============================================================================
def load_master_sheet(master_path):
    master = pd.read_excel(master_path, sheet_name=0)
    master = master.dropna(subset=['Video'])
    keep_cols = ['Video'] + [c for c in GROUP_COLS if c in master.columns]
    master = master[keep_cols].copy()
    master.rename(columns={'Video': 'video'}, inplace=True)
    # Normalise the rogue "_roi2copy" suffix so it collapses with the real ROI
    master['video'] = master['video'].astype(str).str.replace(r'copy$', '', regex=True)
    return master

def get_base_name_from_file(csv_path):
    base = os.path.basename(csv_path)
    if base.startswith("aug_"):
        base = base[4:]
    if base.endswith(".csv"):
        base = base[:-4]
    return base

def extract_video_name_from_filename(basename, master_videos):
    matches = [mv for mv in master_videos if basename.startswith(mv)]
    return max(matches, key=len) if matches else None

# =============================================================================
# COLLECT DATA
# =============================================================================
master = load_master_sheet(MASTER_SHEET_PATH)
master_videos = set(master['video'])

all_csv = glob(os.path.join(AUGMENTED_CSV_FOLDER, '*.csv'))
valid_csv = []
for f in all_csv:
    try:
        df_sample = pd.read_csv(f, nrows=5)
        if 'signed_angle_to_SS' in df_sample.columns:
            valid_csv.append(f)
    except Exception:
        continue

polar_data = []
for csv_path in valid_csv:
    basename = get_base_name_from_file(csv_path)
    video_name = extract_video_name_from_filename(basename, master_videos)
    if video_name is None:
        match = re.search(r'(MVI_\d+_roi\d+)', basename)
        video_name = match.group(1) if match else None
        if not video_name:
            continue

    if video_name in master_videos:
        row = master[master['video'] == video_name].iloc[0]
        group_vals = {gc: row[gc] if pd.notna(row[gc]) else 'unknown' for gc in GROUP_COLS}
    else:
        group_vals = {gc: 'unknown' for gc in GROUP_COLS}

    df = pd.read_csv(csv_path)
    if 'signed_angle_to_SS' not in df.columns:
        continue

    for stim, col in [('SS', 'signed_angle_to_SS'), ('FS', 'signed_angle_to_FS')]:
        angles = df[col].dropna().values
        for ang in angles:
            polar_data.append({
                'video': video_name,
                'stimulus': stim,
                'signed_angle': ang,
                **group_vals
            })

if not polar_data:
    raise RuntimeError("No valid angle data collected.")

polar_df = pd.DataFrame(polar_data)

# ---- diagnostics ----
print("=== Master Treatment distribution ===")
print(master['Treatment'].value_counts(dropna=False))
print()
print("=== Videos per treatment (raw, ROI-aware) ===")
print(polar_df.groupby('Treatment')['video'].nunique())
print()

treatment_df = polar_df[polar_df['Treatment'] != 'unknown'].copy()

# =============================================================================
# PUBLICATION POLAR PLOTS (2x2 GRID) — seconds per video
# =============================================================================
def plot_treatment_polar_publication(df, output_folder, frame_rate=FRAME_RATE):
    groups = sorted(df['Treatment'].unique())

    fig, axes = plt.subplots(2, 2, figsize=(14, 13), subplot_kw={'projection': 'polar'})
    axes = axes.flatten()

    colors     = {'SS': '#CC79A7', 'FS': '#009E73'}
    linewidths = {'SS': 4.0,      'FS': 4}
    markers    = {'SS': '.',      'FS': '.'}

    n_bins = 36
    bin_edges     = np.linspace(-np.pi, np.pi, n_bins + 1)
    theta_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    theta_closed  = np.r_[theta_centers, theta_centers[:1] + 2 * np.pi]

    # ---- number of unique videos per group (raw, ROI-aware) ----
    n_videos_per_group = df.groupby('Treatment')['video'].nunique().to_dict()

    # ---- global radial upper limit (seconds per video) ----
    all_max = 0
    for g in groups:
        n_vid = n_videos_per_group.get(g, 1) or 1
        for stim in ['SS', 'FS']:
            vals = df[(df['Treatment'] == g) & (df['stimulus'] == stim)]['signed_angle'].values
            if len(vals) > 0:
                counts, _ = np.histogram(np.radians(vals), bins=bin_edges)
                all_max = max(all_max, counts.max() / (n_vid * frame_rate))

    if all_max <= 0:
        all_max = 1
    magnitude = 10 ** np.floor(np.log10(all_max))
    y_limit   = np.ceil(all_max / magnitude) * magnitude
    step      = y_limit / 5.0
    tick_radii = [round(step * i, 10) for i in range(1, 6)]

    def _fmt(v):
        return f"{v:g} s"

    for idx, g in enumerate(groups[:4]):
        ax = axes[idx]
        n_vid = n_videos_per_group.get(g, 1) or 1

        for stim in ['SS', 'FS']:
            vals = df[(df['Treatment'] == g) & (df['stimulus'] == stim)]['signed_angle'].values
            if len(vals) == 0:
                continue
            counts, _ = np.histogram(np.radians(vals), bins=bin_edges)

            # seconds per video, per 5-degree bin
            sec_per_video = counts / (n_vid * frame_rate)
            sec_closed    = np.r_[sec_per_video, sec_per_video[:1]]

            ax.plot(theta_closed, sec_closed,
                    color=colors[stim],
                    linewidth=linewidths[stim],
                    marker=markers[stim],
                    markersize=5,
                    markerfacecolor=colors[stim],
                    markeredgecolor=colors[stim],
                    alpha=0.9,
                    label=stim)

        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)
        ax.set_ylim(0, y_limit)

        ax.set_thetagrids(np.arange(0, 360, 45),
                          labels=['0°','45°','90°','135°','180°','225°','270°','315°'],
                          fontsize=22)
        ax.tick_params(axis='x', pad=10)

        ax.set_rgrids(tick_radii, labels=[_fmt(v) for v in tick_radii],
                      angle=0, fontsize=14)
        ax.set_rlabel_position(0)
        ax.tick_params(axis='y', pad=2)

        ax.set_title(f"{g}\n(n = {n_vid})", fontsize=22, fontweight='bold', pad=18)
        ax.grid(True, alpha=0.4, linestyle=':', linewidth=1)
        ax.yaxis.set_major_locator(mticker.FixedLocator(tick_radii))

    for idx in range(len(groups), 4):
        fig.delaxes(axes[idx])

    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], color=colors['SS'], lw=linewidths['SS'], label='SS'),
        Line2D([0], [0], color=colors['FS'], lw=linewidths['FS'], label='FS'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=2,
               frameon=False, fontsize=22, bbox_to_anchor=(0.5, 0.0))

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.10, right=0.88, top=0.90, hspace=0.50, wspace=0.25)

    out_path = os.path.join(output_folder, "treatment_polar_plot_publication_v1.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(out_path.replace('.png', '.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Polar 2x2 plot saved to: {out_path}")
    print("Videos per treatment group:", n_videos_per_group)

# Run generation
plot_treatment_polar_publication(treatment_df, OUTPUT_PLOTS_FOLDER)

=== Master Treatment distribution ===
Treatment
SH FD    18
SH AL    11
GH AL    10
GH FD    10
Name: count, dtype: int64

=== Videos per treatment (raw, ROI-aware) ===
Treatment
GH AL    10
GH FD    10
SH AL    11
SH FD    17
Name: video, dtype: int64

Polar 2x2 plot saved to: polar_plots_publication/treatment_polar_plot_publication_v1.png
Videos per treatment group: {'GH AL': 10, 'GH FD': 10, 'SH AL': 11, 'SH FD': 17}


In [7]:
# =============================================================================
# DIAGNOSTIC — why are videos being dropped?
# =============================================================================
print(f"Total CSVs found:        {len(all_csv)}")
print(f"CSVs with angle column:  {len(valid_csv)}")

# --- Step 1: what does the filename -> video_name matching look like? ---
matched_names = []
unmatched_basenames = []

for csv_path in valid_csv:
    basename = get_base_name_from_file(csv_path)
    video_name = extract_video_name_from_filename(basename, master_videos)
    if video_name is None:
        match = re.search(r'(MVI_\d+_roi\d+)', basename)
        video_name = match.group(1) if match else None
    if video_name is None:
        unmatched_basenames.append(basename)
    else:
        matched_names.append(video_name)

print(f"CSVs matched to a video name:   {len(matched_names)}")
print(f"CSVs NOT matched to any name:   {len(unmatched_basenames)}")
if unmatched_basenames:
    print("\nFirst 20 unmatched basenames (check these):")
    for b in unmatched_basenames[:20]:
        print("   ", b)

# --- Step 2: matched names not present in master sheet? ---
unique_matched = set(matched_names)
not_in_master = sorted(unique_matched - master_videos)
print(f"\nUnique matched names:                 {len(unique_matched)}")
print(f"Matched names NOT in master sheet:    {len(not_in_master)}")
if not_in_master:
    print("First 20 names not in master:")
    for n in not_in_master[:20]:
        print("   ", n)

# --- Step 3: what do the master video IDs actually look like? ---
print(f"\nMaster sheet video IDs (first 20):")
for v in sorted(master_videos)[:20]:
    print("   ", v)

# --- Step 4: what do the CSV basenames actually look like? ---
print(f"\nCSV basenames (first 20):")
for csv_path in valid_csv[:20]:
    print("   ", get_base_name_from_file(csv_path))

Total CSVs found:        50
CSVs with angle column:  49
CSVs matched to a video name:   49
CSVs NOT matched to any name:   0

Unique matched names:                 49
Matched names NOT in master sheet:    0

Master sheet video IDs (first 20):
    MVI_0006_roi1
    MVI_0006_roi2
    MVI_0008_roi1
    MVI_0008_roi2
    MVI_0010_roi1
    MVI_0010_roi2
    MVI_0010_roi2copy
    MVI_0011_roi1
    MVI_0011_roi2
    MVI_0013_roi1
    MVI_0013_roi2
    MVI_0074_roi1
    MVI_0074_roi2
    MVI_0075_roi1
    MVI_0075_roi2
    MVI_0078_roi1
    MVI_0078_roi2
    MVI_0081_roi1
    MVI_0081_roi2
    MVI_0083_roi1

CSV basenames (first 20):
    MVI_0614_roi2DLC_HrnetW32_TCTautotrackMay28shuffle5_detector_050_snapshot_070_cleaned
    MVI_0091_roi1DLC_HrnetW32_TCTautotrackMay28shuffle5_detector_050_snapshot_070_cleaned
    MVI_0616_roi1DLC_HrnetW32_TCTautotrackMay28shuffle5_detector_050_snapshot_070_cleaned
    MVI_0075_roi1DLC_HrnetW32_TCTautotrackMay28shuffle5_detector_050_snapshot_070_cleaned
    MV